In [ ]:
import torch
import torch.nn as nn

N_POINTS = 100

class ContourModel1Improved(nn.Module):

    def __init__(self):
        super().__init__()

        self.encoder = nn.Sequential(

            nn.Linear(400, 512),
            nn.LayerNorm(512),
            nn.GELU(),

            nn.Linear(512, 256),
            nn.LayerNorm(256),
            nn.GELU(),

            nn.Dropout(0.05),

            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.GELU()
        )

        self.middle = nn.Sequential(

            nn.Linear(128, 128),
            nn.GELU(),

            nn.Linear(128, 128),
            nn.GELU()
        )

        self.decoder = nn.Sequential(

            nn.Linear(128, 256),
            nn.LayerNorm(256),
            nn.GELU(),

            nn.Linear(256, 512),
            nn.LayerNorm(512),
            nn.GELU(),

            nn.Linear(512, 200)
        )

    def forward(self, prev_pts, future_pts):

        batch = prev_pts.size(0)

        x = torch.cat(
            [
                prev_pts.view(batch, -1),
                future_pts.view(batch, -1)
            ],
            dim=1
        )

        latent = self.encoder(x)

        latent = latent + self.middle(latent)

        residual = self.decoder(latent)

        return residual.view(batch, N_POINTS, 2)